# streaming_ingest-events-spark-iceberg

## 1. Overview

## 2. Setup

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, StructType, TimestampType

spark = SparkSession.builder.remote("sc://spark-connect:15002").getOrCreate()

## 3. Read

In [ ]:
raw = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "redpanda:9092")
    .option("subscribe", "events")
    .option("startingOffsets", "earliest")
    .load()
)

## 4. Transform

In [ ]:
schema = (
    StructType()
    .add("user_id", StringType())
    .add("event", StringType())
    .add("ts", TimestampType())
)

events = raw.select(
    F.from_json(F.col("value").cast("string"), schema).alias("e")
).select("e.*")

## 5. Write

In [ ]:
query = (
    events.writeStream
    .format("iceberg")
    .outputMode("append")
    .option("checkpointLocation", "s3a://checkpoints/events")
    .toTable("lakehouse.bronze.events")
)

# Run query.awaitTermination() to keep the stream alive for a live Redpanda topic

## 6. Verify

In [ ]:
spark.table("lakehouse.bronze.events").count()